In [76]:
import numpy as np
import time
from collections import defaultdict, deque
import random

random.seed(42)

In [37]:
def load_graph_from_file(filename):
    outbound_links = defaultdict(list)
    inbound_links = defaultdict(list)
    nodes = set()
    
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"): #removing empty lines and comments within dataset
                continue
            
            src, dst = map(int, line.split())
            
            outbound_links[src].append(dst)
            inbound_links[dst].append(src)
            nodes.add(src)
            nodes.add(dst)
            
    for node in nodes:
        outbound_links[node]
        inbound_links[node]
        
    return outbound_links, inbound_links, sorted(nodes)

Compute PageRank using power iteration.
----------

Parameters
----------
outbound_links : dict[node] -> list of outgoing neighbors

inbound_links  : dict[node] -> list of incoming neighbors

nodes          : sorted list of node ids

p              : teleport probability

max_iter       : maximum number of iterations

tol            : L1 convergence tolerance

Returns
-------
ranks          : numpy array of PageRank scores

node_to_idx    : dict mapping node id -> index

iterations     : number of iterations used
 

In [38]:
def pagerank_iterative(outbound_links, inbound_links, nodes, p, max_iter, tol=1e-10):
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}

    ranks = np.full(n, 1.0 / n, dtype=np.float64)

    outdeg = np.zeros(n, dtype=np.int64)
    for node in nodes:
        outdeg[node_to_idx[node]] = len(outbound_links[node])

    teleport = p / n

    for iteration in range(1, max_iter + 1):
        new_ranks = np.zeros(n, dtype=np.float64)

        # rank mass from dangling nodes
        dangling_mass = 0.0
        for node in nodes:
            i = node_to_idx[node]
            if outdeg[i] == 0:
                dangling_mass += ranks[i]

        dangling_contrib = (1 - p) * dangling_mass / n

        # update each node
        for u in nodes:
            ui = node_to_idx[u]
            incoming_sum = 0.0

            for v in inbound_links[u]:
                vi = node_to_idx[v]
                if outdeg[vi] > 0:
                    incoming_sum += ranks[vi] / outdeg[vi]

            new_ranks[ui] = (1 - p) * incoming_sum + dangling_contrib + teleport

        new_ranks /= new_ranks.sum()

        if np.linalg.norm(new_ranks - ranks, 1) < tol:
            return new_ranks, node_to_idx, iteration

        ranks = new_ranks

    return ranks, node_to_idx, max_iter

In [39]:
def top_k_pagerank(nodes, ranks, k=10):
    pairs = list(zip(nodes, ranks))
    pairs.sort(key=lambda x: x[1], reverse=True)
    return pairs[:k]


def save_pagerank_output(filename, nodes, ranks):
    pairs = list(zip(nodes, ranks))
    pairs.sort(key=lambda x: x[1], reverse=True)

    with open(filename, "w", encoding="utf-8") as f:
        for node, score in pairs:
            f.write(f"{score:.12f}\t{node}\n")

Compare the numerical result with the closed form algorithm
--

Closed Form PageRank Algorithm:

r = (p/n) * [I - (1-p)M]^{-1} * 1

In [81]:
def sample_subgraph(outbound_links, inbound_links, nodes, sample_size):
    seed = random.choice(nodes) #select nodes at random instead of just taking first 200
    visited = set([seed])
    queue = deque([seed])

    while queue and len(visited) < sample_size:
        u = queue.popleft()

        neighbors = outbound_links[u] + inbound_links[u]
        for v in neighbors:
            if v not in visited:
                visited.add(v)
                queue.append(v)
                if len(visited) >= sample_size:
                    break

    sampled_nodes = sorted(visited)
    sampled_set = set(sampled_nodes)

    sub_out = defaultdict(list)
    sub_in = defaultdict(list)

    for u in sampled_nodes:
        for v in outbound_links[u]:
            if v in sampled_set:
                sub_out[u].append(v)
                sub_in[v].append(u)

    for u in sampled_nodes:
        sub_out[u]
        sub_in[u]

    return sub_out, sub_in, sampled_nodes

def build_transition_matrix_dense(outbound_links, nodes):
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    M = np.zeros((n, n), dtype=np.float64)

    for src in nodes:
        j = node_to_idx[src]
        outs = outbound_links[src]

        if len(outs) == 0:
            M[:, j] = 1.0 / n
        else:
            prob = 1.0 / len(outs)
            for dst in outs:
                i = node_to_idx[dst]
                M[i, j] += prob

    return M, node_to_idx


def pagerank_closed_form(M, p):
    n = M.shape[0]
    I = np.eye(n)
    ones = np.ones(n)

    r = (p / n) * np.linalg.solve(I - (1 - p) * M, ones)
    r /= r.sum()
    return r


def compare_iterative_vs_closed_form(outbound_links, inbound_links, nodes, p, sample_size, max_iter, tol):
    sub_out, sub_in, sub_nodes = sample_subgraph(outbound_links, inbound_links, nodes, sample_size)

    iter_ranks, iter_idx, iterations = pagerank_iterative(
        sub_out, sub_in, sub_nodes, p=p, max_iter=max_iter, tol=tol
    )

    M_dense, dense_idx = build_transition_matrix_dense(sub_out, sub_nodes)
    closed_ranks = pagerank_closed_form(M_dense, p=p)

    l1_error = np.linalg.norm(iter_ranks - closed_ranks, 1)
    max_error = np.max(np.abs(iter_ranks - closed_ranks))
    corr = np.corrcoef(iter_ranks, closed_ranks)[0, 1]

    print("\nComparison: iterative vs closed form")
    print("Sample size:", sample_size)
    print("Iterations :", iterations)
    print("L1 error   :", l1_error)
    print("Max error  :", max_error)
    print("Correlation:", corr)

    print("\nFirst 10 sampled nodes:")
    print(f"{'Node':>10} {'Iterative':>15} {'ClosedForm':>15} {'AbsDiff':>15}")
    for node in sub_nodes[:10]:
        i = iter_idx[node]
        j = dense_idx[node]
        diff = abs(iter_ranks[i] - closed_ranks[j])
        print(f"{node:>10} {iter_ranks[i]:15.10f} {closed_ranks[j]:15.10f} {diff:15.10e}")

Loading the Data from the Google Dataset

In [54]:
outbound_links, inbound_links, nodes = load_graph_from_file("Data/web-Google.txt")
p = 0.15 #setting the "random surfer's" teleport probability
tol = 1e-12 #setting the tolerance for all tests

In [42]:
print("Nodes:", len(nodes))
print("Edges:", sum(len(v) for v in outbound_links.values()))
print("\t") #add a new line
print("Structure of the first 10 nodes:")
for node in nodes[:10]:
    print(node, "->", outbound_links[node])

Nodes: 875713
Edges: 5105039
	
Structure of the first 10 nodes:
0 -> [11342, 824020, 867923, 891835]
1 -> [53051, 203402, 223236, 276233, 552600, 569212, 635575, 748615, 862566, 893884]
2 -> [30957, 357310, 423174, 430119, 462435, 472889, 565424, 581609, 597621, 644135, 858904]
3 -> []
4 -> [44695, 62391, 79146, 173976, 670449, 746182, 795253]
5 -> [39733, 219773, 300279, 535142, 579655, 581741, 608321]
6 -> [119755, 188708, 668992]
7 -> [19576, 43343, 57818, 75039, 86833, 174836, 226374, 230497, 358176, 487767, 507854, 514170, 694921, 743269, 747035, 877041]
8 -> [55948, 846855]
9 -> [504263, 704303, 721062, 780043]


As it can be observed above, the number of nodes and edges matches the information that is provided by https://hunglvosu.github.io/posts/2020/07/PA3/.

Outbound links will be used to compute the outdegree and the inbound links will be used to compute the PageRank contributions.

In [43]:
print("Sample outbound links:")
for node in nodes[:5]:
    print(node, "->", outbound_links[node])
    
print("\t") #add a new line

print("Sample inbound links:")
for node in nodes[:5]:
    print(node, "<-", inbound_links[node])

Sample outbound links:
0 -> [11342, 824020, 867923, 891835]
1 -> [53051, 203402, 223236, 276233, 552600, 569212, 635575, 748615, 862566, 893884]
2 -> [30957, 357310, 423174, 430119, 462435, 472889, 565424, 581609, 597621, 644135, 858904]
3 -> []
4 -> [44695, 62391, 79146, 173976, 670449, 746182, 795253]
	
Sample inbound links:
0 <- [11342, 824020, 867923, 891835, 635343, 600594, 366080, 500627, 835220, 543999, 830675, 95656, 38716, 322178, 387543, 638706, 645018, 856657, 11927, 180285, 395856, 450550, 824375, 143557, 15901, 108113, 136131, 320258, 335113, 512812, 522510, 854209, 857934, 859247, 857527, 828241, 29546, 535748, 30281, 468354, 62929, 31300, 553829, 136593, 414038, 428822, 523684, 684417, 760842, 765654, 795148, 815602, 72432, 96460, 160114, 193375, 293748, 365423, 403327, 491156, 548921, 609210, 635163, 636635, 662982, 692491, 721453, 728417, 867959, 43929, 115472, 494671, 46274, 877381, 48192, 50822, 554441, 56910, 68314, 71878, 283476, 701455, 821092, 73631, 483644, 6449

The above sample graph structure for the inbound and outbound links are printed to verify correctness.

In [73]:
start = time.time()

max_iter=100

ranks, node_to_idx, iterations = pagerank_iterative(
    outbound_links, inbound_links, nodes, p, max_iter, tol
)

print("Iterations:", iterations)
print("Rank sum:", ranks.sum())

top10 = top_k_pagerank(nodes, ranks, k=10)
print("\nTop 10 pages by PageRank:")
for i, (node, score) in enumerate(top10, start=1):
    print(f"{i}. Node {node}: {score:.12f}")

save_pagerank_output("PR_web-Google_Output.txt", nodes, ranks)

print(f"\nTotal runtime: {time.time() - start:.3f} seconds")

Iterations: 100
Rank sum: 1.0000000000000009

Top 10 pages by PageRank:
1. Node 597621: 0.000923724940
2. Node 41909: 0.000921131235
3. Node 163075: 0.000904004421
4. Node 537039: 0.000898831798
5. Node 384666: 0.000786892436
6. Node 504140: 0.000765116046
7. Node 486980: 0.000724940300
8. Node 605856: 0.000717955259
9. Node 32163: 0.000712571843
10. Node 558791: 0.000709185929

Total runtime: 425.123 seconds


In [92]:
sample_size = 20000
max_iter = 500

compare_iterative_vs_closed_form(
    outbound_links, inbound_links, nodes, p, sample_size, max_iter, tol
)


Comparison: iterative vs closed form
Sample size: 20000
Iterations : 144
L1 error   : 1.1581621417650935e-12
Max error  : 4.475922670693633e-14
Correlation: 0.9999999999999998

First 10 sampled nodes:
      Node       Iterative      ClosedForm         AbsDiff
         1    0.0000754678    0.0000754678 7.3983245745e-17
       106    0.0001674957    0.0001674957 2.0930522940e-16
       192    0.0000302490    0.0000302490 3.9302328753e-17
       232    0.0000129578    0.0000129578 8.6227954030e-19
       280    0.0000505552    0.0000505552 7.1083004934e-18
       297    0.0000273980    0.0000273980 9.5409791179e-18
       299    0.0000272328    0.0000272328 1.5551524912e-18
       314    0.0000185464    0.0000185464 3.3271454168e-18
       360    0.0000094796    0.0000094796 5.7598240413e-19
       441    0.0000264183    0.0000264183 2.5309344464e-18
